# 이상 터빈 분석
- 발전량 <= 0 인 터빈은 이상이 있다고 판단
- 이상 이유 분석 위한 노트북
- 오류 or 약풍, 강풍 or 점검

# 라이브러리

In [10]:
import pandas as pd
import numpy as np

# 결측치 처리된 데이터 불러오기

In [3]:
# 그룹1
train_labels_group1 = pd.read_csv('../../data/after_missing/train_labels_group1_missing.csv', encoding = 'utf-8-sig')

# 그룹2
train_labels_group2 = pd.read_csv('../../data/after_missing/train_labels_group2_missing.csv', encoding = 'utf-8-sig')

# 그룹3
train_labels_group3 = pd.read_csv('../../data/after_missing/train_labels_group3_missing.csv', encoding = 'utf-8-sig')

In [4]:
# SCADA 실측데이터
scada_unison = pd.read_csv('../../open/train/scada_unison_train.csv', encoding = 'utf-8-sig')
scada_vestas = pd.read_csv('../../open/train/scada_vestas_train.csv', encoding = 'utf-8-sig')

# 터빈 정지인 시각을 자세히 살펴보기
- 터빈 정지로 label = 0인 경우
- 몇 개의 터빈만 멈춰서 발전량이 적은 경우
- 휴리스틱하게 판단하기 위해 label 값 | 각 터빈 발전량 | 활동 터빈 | 정지 터빈 | 풍속 | 풍향 을 데이터프레임으로 만들자. (그룹별로)

In [5]:
# 0. 그룹 정의
#    CAP_MODE = 'rated'  -> 3600 / 4200 (설비용량 기준)
#    CAP_MODE = 'per10m' -> 600 / 700   (10분 에너지 물리 상한)
# ---------------------------------------------------------------------
CAP_MODE = 'rated'

GROUP_DEF = {
    1: dict(src='vestas', wtgs=[f'{i:02d}' for i in range(1, 7)],  cap_rated=3600, cap_10m=600),
    2: dict(src='vestas', wtgs=[f'{i:02d}' for i in range(7, 13)], cap_rated=3600, cap_10m=600),
    3: dict(src='unison', wtgs=[f'{i:02d}' for i in range(1, 6)],  cap_rated=4200, cap_10m=700),
}

SCADA  = {'vestas': scada_vestas, 'unison': scada_unison}
LABELS = {1: train_labels_group1, 2: train_labels_group2, 3: train_labels_group3}

In [6]:
# 1. 10분 슬롯 시각 -> 대응 정시(집계 종료 시각)
#    14:10 ~ 15:00 의 6개 슬롯이 15:00 에 대응한다.
# ---------------------------------------------------------------------
def slot_to_hour(ts: pd.Series) -> pd.Series:
    return (ts - pd.Timedelta(seconds=1)).dt.floor('h') + pd.Timedelta(hours=1)

In [7]:
# 2. 그룹별 정지 분석 테이블 생성
# ---------------------------------------------------------------------
def build_stop_table(group_id: int, cap_mode: str = CAP_MODE, only_stop_rows: bool = True):
    g   = GROUP_DEF[group_id]
    cap = g['cap_rated'] if cap_mode == 'rated' else g['cap_10m']
    src, wtgs = g['src'], g['wtgs']

    pw_cols = [f'{src}_wtg{w}_power_kw10m' for w in wtgs]
    ws_cols = [f'{src}_wtg{w}_ws' for w in wtgs]
    wd_cols = [f'{src}_wtg{w}_wd' for w in wtgs]

    sc = SCADA[src][['kst_dtm'] + pw_cols + ws_cols + wd_cols].copy()
    sc['kst_dtm'] = pd.to_datetime(sc['kst_dtm'])

    P = sc[pw_cols].to_numpy(dtype=float)

    # --- 정지 판정: 발전량 <= 0  또는  발전량 > CAP ---
    #     NaN 은 '판정 불가'로 두고 정지에서 제외
    stopped  = ((P <= 0) | (P > cap)) & ~np.isnan(P)
    nan_mask = np.isnan(P)
    active   = ~stopped & ~nan_mask

    names = np.array([f'wtg{w}' for w in wtgs])

    out = pd.DataFrame({'slot_dtm': sc['kst_dtm']})
    out['kst_dtm'] = slot_to_hour(sc['kst_dtm'])          # 대응 정시 (label 매칭 키)

    # --- 터빈별 발전량 ---
    for w, c in zip(wtgs, pw_cols):
        out[f'pw_wtg{w}'] = sc[c].to_numpy(dtype=float)

    # --- 활동 / 정지 / NaN 개수 및 목록 ---
    out['n_active']     = active.sum(axis=1)
    out['n_stopped']    = stopped.sum(axis=1)
    out['n_nan']        = nan_mask.sum(axis=1)
    out['active_wtgs']  = [', '.join(names[r]) for r in active]
    out['stopped_wtgs'] = [', '.join(names[r]) for r in stopped]

    # --- 터빈별 풍속 / 풍향 ---
    for w, c in zip(wtgs, ws_cols):
        out[f'ws_wtg{w}'] = sc[c].to_numpy(dtype=float)
    for w, c in zip(wtgs, wd_cols):
        out[f'wd_wtg{w}'] = sc[c].to_numpy(dtype=float)

    # --- label 붙이기 ---
    lb = LABELS[group_id].copy()
    tcol = [c for c in lb.columns if 'dtm' in c.lower()][0]
    vcol = f'kpx_group_{group_id}'
    lb = lb[[tcol, vcol]].rename(columns={tcol: 'kst_dtm', vcol: 'label'})
    lb['kst_dtm'] = pd.to_datetime(lb['kst_dtm'])
    out = out.merge(lb, on='kst_dtm', how='left')

    # --- 정지 터빈이 1대 이상인 슬롯만 남김 ---
    if only_stop_rows:
        out = out[out['n_stopped'] > 0].copy()

    # --- 컬럼 순서: 시각 | label | 10분슬롯 | 터빈발전량 | 활동/정지 | 풍속 | 풍향 ---
    front = ['kst_dtm', 'label', 'slot_dtm',
             *[f'pw_wtg{w}' for w in wtgs],
             'n_active', 'n_stopped', 'n_nan', 'active_wtgs', 'stopped_wtgs',
             *[f'ws_wtg{w}' for w in wtgs],
             *[f'wd_wtg{w}' for w in wtgs]]
    rest = [c for c in out.columns if c not in front]
    return out[front + rest].sort_values(['kst_dtm', 'slot_dtm']).reset_index(drop=True)

In [8]:
# 3. 실행
# ---------------------------------------------------------------------
stop_group1 = build_stop_table(1)
stop_group2 = build_stop_table(2)
stop_group3 = build_stop_table(3)

for gid, df in zip([1, 2, 3], [stop_group1, stop_group2, stop_group3]):
    g = GROUP_DEF[gid]
    cap = g['cap_rated'] if CAP_MODE == 'rated' else g['cap_10m']
    print(f'[그룹{gid}] {g["src"]} {g["wtgs"][0]}~{g["wtgs"][-1]}호기 | CAP={cap} | '
          f'정지슬롯 {len(df):,}행 / 정시 {df["kst_dtm"].nunique():,}개')

[그룹1] vestas 01~06호기 | CAP=3600 | 정지슬롯 74,262행 / 정시 14,304개
[그룹2] vestas 07~12호기 | CAP=3600 | 정지슬롯 65,742행 / 정시 13,402개
[그룹3] unison 01~05호기 | CAP=4200 | 정지슬롯 63,041행 / 정시 12,110개


In [9]:
stop_group1.head()

,kst_dtm,label,slot_dtm,pw_wtg01,pw_wtg02,pw_wtg03,pw_wtg04,pw_wtg05,pw_wtg06,n_active,...,ws_wtg03,ws_wtg04,ws_wtg05,ws_wtg06,wd_wtg01,wd_wtg02,wd_wtg03,wd_wtg04,wd_wtg05,wd_wtg06
0,2022-01-01 12:00:00,940.105,2022-01-01 11:30:00,12.0,16.0,31.0,36.0,49.0,-3.0,5,...,4.5,4.7,4.9,3.4,309.7,259.6,281.5,299.5,271.0,293.1
1,2022-01-01 12:00:00,940.105,2022-01-01 12:00:00,3.0,6.0,14.0,4.0,13.0,-4.0,5,...,4.2,3.9,3.8,3.2,310.5,264.2,272.7,296.0,280.9,275.2
2,2022-01-01 13:00:00,520.674,2022-01-01 12:10:00,2.0,0.0,8.0,0.0,-5.0,0.0,2,...,4.0,3.6,2.7,3.4,303.4,267.5,278.9,292.7,281.2,254.6
3,2022-01-01 13:00:00,520.674,2022-01-01 12:20:00,1.0,-5.0,1.0,-2.0,-5.0,0.0,2,...,3.7,3.3,2.9,3.6,295.4,256.3,267.0,282.5,268.1,267.9
4,2022-01-01 13:00:00,520.674,2022-01-01 12:30:00,7.0,7.0,4.0,2.0,-4.0,0.0,4,...,3.9,4.0,2.8,3.4,288.5,244.8,263.3,272.7,258.4,259.9


### 파일 저장
stop_group1.to_csv('../../reference/turbine_stop/turbine_stop_group1.csv', index = False, encoding = 'utf-8-sig')
stop_group2.to_csv('../../reference/turbine_stop/turbine_stop_group2.csv', index = False, encoding = 'utf-8-sig')
stop_group3.to_csv('../../reference/turbine_stop/turbine_stop_group3.csv', index = False, encoding = 'utf-8-sig')

# 터빈 정지 데이터 처리

# 처리 후 clean 데이터로 저장
- 이 데이터로 함수 적용시켜 파생변수 생성 및 모델 학습